# M4b — label-free prompt H2 null test

Reproduces the headline tables from `docs/insights/m4b_label_free.md`.
Loads the label-free predictions and pairs them against M2's labeled
open-prompt rows on the same `(obj, bg, cue, seed)` to derive the
per-label PMR contribution.

Depends on outputs already on disk:
- `outputs/label_free_20260425-031430_315c5318/` (this milestone)
- `outputs/mvp_full_20260424-094103_8ae1fa3d/` (M2 baseline)

If the label-free run hasn't been executed yet, run
`uv run python scripts/02_run_inference.py --config configs/label_free.py --stimulus-dir inputs/mvp_full_20260424-093926_e9d79da3` first (~13 min on H200).

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

LF_DIR = PROJECT_ROOT / 'outputs' / 'label_free_20260425-031430_315c5318'
M2_DIR = PROJECT_ROOT / 'outputs' / 'mvp_full_20260424-094103_8ae1fa3d'

LF = pd.read_parquet(LF_DIR / 'predictions_scored.parquet')
M2 = pd.read_parquet(M2_DIR / 'predictions_scored.parquet')
M2_OPEN = M2[M2.prompt_variant == 'open']
print(f'label-free: n={len(LF)}, prompt={LF.prompt_variant.unique()[0]}, label={LF.label.unique()[0]}')
print(f'M2 open:   n={len(M2_OPEN)}, labels={sorted(M2_OPEN.label.unique())}')

label-free: n=480, prompt=open_no_label, label=_nolabel
M2 open:   n=1440, labels=['ball', 'circle', 'planet']


## Overall PMR / GAR — label-free vs M2 by label

In [2]:
lf_overall = pd.DataFrame({
    'pmr': [LF.pmr.mean()], 'gar': [LF.gar.mean()],
    'hold_still': [LF.hold_still.mean()], 'abstract_reject': [LF.abstract_reject.mean()],
}, index=['_nolabel (LF)'])
m2_overall = M2_OPEN.groupby('label')[['pmr','gar','hold_still','abstract_reject']].mean()
pd.concat([m2_overall, lf_overall]).round(3)

,pmr,gar,hold_still,abstract_reject
ball,0.954,0.706,0.062,0.000
circle,0.883,0.753,0.160,0.002
planet,0.954,0.319,0.081,0.004
_nolabel (LF),0.948,0.728,0.127,0.002


## Headline: paired PMR delta — `PMR(label) − PMR(_nolabel)`

Pairing key: `sample_id` (encodes obj × bg × cue × seed). For each M2
label, the delta against the label-free baseline is the same-image,
same-seed PMR difference attributable to the label token.

If a label simply restates what the model already sees (visual default),
the delta is ~0. A non-zero delta indicates active language-prior
modulation.

In [3]:
lf_pmr = LF.groupby('sample_id').pmr.mean().rename('_nolabel')
rows = []
for lab in ['ball', 'circle', 'planet']:
    m2_pmr = M2_OPEN[M2_OPEN.label == lab].groupby('sample_id').pmr.mean().rename(lab)
    j = pd.concat([lf_pmr, m2_pmr], axis=1).dropna()
    delta = j[lab] - j._nolabel
    rows.append({'label': lab, 'mean_delta': float(delta.mean()), 'n': len(j)})
pd.DataFrame(rows).set_index('label').round(3)

,mean_delta,n
label,,
ball,0.006,480
circle,-0.065,480
planet,0.006,480


Reading: `ball` and `planet` ≈ 0, `circle` = −0.065. The +15 pp "ball
vs circle" gap from M2 is **circle suppression**, not ball enhancement.

## Per-object-level structure

In [4]:
lf_obj = LF.groupby('object_level').pmr.mean().rename('_nolabel')
m2_obj = M2_OPEN.groupby(['object_level', 'label']).pmr.mean().unstack('label')
t = pd.concat([m2_obj, lf_obj], axis=1)
t['ball − _nolabel']   = t['ball'] - t['_nolabel']
t['circle − _nolabel'] = t['circle'] - t['_nolabel']
t.round(3)

,ball,circle,planet,_nolabel,ball − _nolabel,circle − _nolabel
object_level,,,,,,
filled,0.950,0.892,0.958,0.933,0.017,-0.042
line,0.950,0.850,0.917,0.942,0.008,-0.092
shaded,0.933,0.892,0.975,0.942,-0.008,-0.050
textured,0.983,0.900,0.967,0.975,0.008,-0.075


Pattern: circle suppression scales with image abstraction
(line: −9.2 pp; filled: −4.2 pp). Image-side dual of H4.

## Per-cue-level structure

In [5]:
lf_cue = LF.groupby('cue_level').pmr.mean().rename('_nolabel')
m2_cue = M2_OPEN.groupby(['cue_level', 'label']).pmr.mean().unstack('label')
t = pd.concat([m2_cue, lf_cue], axis=1)
t['circle − _nolabel'] = t['circle'] - t['_nolabel']
t.round(3)

,ball,circle,planet,_nolabel,circle − _nolabel
cue_level,,,,,
both,0.992,0.992,0.992,0.983,0.008
cast_shadow,1.000,0.850,0.958,0.967,-0.117
motion_arrow,1.000,1.000,0.992,1.000,0.000
none,0.825,0.692,0.875,0.842,-0.150


`motion_arrow` cue overrides circle suppression (+0.000); cue=`none`
gives the largest suppression (−15 pp).

## Cleanest figure — `line/blank/none` (fully abstract image)

Only at this cell does the visual ambiguity expose all three label
effects orthogonally.

In [6]:
rows = []
cell_filter = lambda d: d[(d.object_level == 'line') & (d.bg_level == 'blank') & (d.cue_level == 'none')]
rows.append(('_nolabel', cell_filter(LF)))
for lab in ['ball', 'circle', 'planet']:
    rows.append((lab, cell_filter(M2_OPEN[M2_OPEN.label == lab])))
out = pd.DataFrame([
    {'label': name, 'pmr': sub.pmr.mean(), 'hold_still': sub.hold_still.mean(),
     'gar': sub.gar.mean(), 'n': len(sub)}
    for name, sub in rows
]).set_index('label')
out.round(2)

,pmr,hold_still,gar,n
label,,,,
_nolabel,0.4,0.2,NaN,10
ball,0.4,0.6,NaN,10
circle,0.1,1.0,NaN,10
planet,0.7,0.3,NaN,10


- `_nolabel`, `ball`: same PMR (0.40); `ball` shifts regime to static
  (hold_still 0.20 → 0.60).
- `circle`: PMR drops to 0.10, hold_still 1.00 — full abstract override.
- `planet`: PMR rises to 0.70 — only label that genuinely *adds* physics
  over the visual default (orbital prior).

## Methodological check — visual-token captures are prompt-independent

M4's `lm_hidden_*` captures at visual-token positions are bit-for-bit
identical between the M2 (with label) and label-free runs, because
Qwen2.5-VL puts image tokens before question text under causal attention.
The label cannot influence visual-token hidden states.

This means the M4 logit lens "switching layer at L5" is a structural
artefact of the capture point, not evidence of label-independent LM
commitment.

In [7]:
from safetensors.torch import load_file

m2_act = load_file(str(M2_DIR / 'activations' / 'line_blank_none_fall_000.safetensors'))
lf_act = load_file(str(LF_DIR / 'activations' / 'line_blank_none_fall_000.safetensors'))
for k in sorted(m2_act):
    if k.startswith('lm_hidden_'):
        diff = (m2_act[k].float() - lf_act[k].float()).abs().max().item()
        print(f'  {k}: shape={tuple(m2_act[k].shape)}, max |M2 − LF| = {diff:.6f}')

  lm_hidden_10: shape=(324, 3584), max |M2 − LF| = 0.000000
  lm_hidden_15: shape=(324, 3584), max |M2 − LF| = 0.000000
  lm_hidden_20: shape=(324, 3584), max |M2 − LF| = 0.000000
  lm_hidden_25: shape=(324, 3584), max |M2 − LF| = 0.000000
  lm_hidden_5: shape=(324, 3584), max |M2 − LF| = 0.000000


## Sample raw responses (T=0.7)

In [8]:
for cell in [('line', 'blank', 'none'), ('textured', 'blank', 'none'),
             ('line', 'blank', 'both'), ('textured', 'ground', 'both')]:
    sub = LF[(LF.object_level == cell[0]) & (LF.bg_level == cell[1]) & (LF.cue_level == cell[2])]
    if len(sub) == 0:
        continue
    r = sub.iloc[0]
    print(f'  {"/".join(cell):28s}: {r.raw_text[:160]}')

  line/blank/none             : The image shows a simple outline of a circle, and it is unlikely to change as it appears to be a static image.
  textured/blank/none         : The image shows a bowling ball with holes, and it's likely to roll if thrown or pushed.
  line/blank/both             : The circle is likely to fall towards the oval shape due to gravity.
  textured/ground/both        : The ball is about to collide with the surface due to gravity pulling it downward.
